# PhyloDater 快速入门

> **语言：** [English (canonical)](01_quickstart.ipynb) · 中文（本文档）

十分钟走一遍真实运行：先用命令行，再用 Python API，最后跑随包发布的“带真值”精度核对。

**开始之前**：PhyloDater 是编排层，不自带定年引擎。请先安装至少一个引擎，并检查当前环境里
到底有哪些（本笔记本最后一节就是做这件事的）。逐个引擎的安装说明（含已从 conda / PyPI 下架
的引擎）见 [docs/EXTERNAL_TOOLS_CN.md](../docs/EXTERNAL_TOOLS_CN.md)。
请从已激活环境的 kernel 运行本笔记本（其 `PATH` 需包含这些引擎），否则对应方法会报
`Environment not available` 并跳过。

目录：[1 输入](#1-输入) · [2 命令行](#2-用命令行运行定年分析) · [3 Python API](#3-python-api) ·
[4 精度核对](#4-对照真值核查精度) · [5 可用性](#5-哪些方法可以在本环境运行) · [6 延伸阅读](#6-延伸阅读)

请**在 `tutorials/` 目录下**运行各单元格（代码中的路径以此为基准）。

## 1. 输入

需要三个文件：一棵树、一个序列比对和一个校准配置。仓库自带示例，所以这里不需要凭空编造数据：

- `examples/tree.nwk`、`examples/alignment.fasta`、`examples/calibrations.yaml`
- `test-data/benchmark/…` —— 同样三类文件，外加 `truth.yaml`，其中记录了已知的分歧时间，
  供 §4 的精度核对使用。

In [ ]:
from pathlib import Path

repo = Path().resolve().parent          # 请在 tutorials/ 目录下运行本笔记本
for rel in ["examples/tree.nwk", "examples/alignment.fasta",
            "examples/calibrations.yaml",
            "test-data/benchmark/tree.nwk", "test-data/benchmark/truth.yaml"]:
    p = repo / rel
    print(f"{'ok ' if p.exists() else 'MISSING'} {rel}")
print()
print((repo / "examples" / "tree.nwk").read_text().strip())
print()
print((repo / "examples" / "calibrations.yaml").read_text().strip()[:400])

## 2. 用命令行运行定年分析

`phylodater dating` 是入口命令。`-t/-s/-c` 是上面那三个输入，`-o` 是输出目录，`--method`
取七个受支持引擎之一，`--seed` 让带随机性的引擎可复现。PATHd8 既确定又快速，很适合作为
第一次运行。

In [ ]:

import subprocess, sys, shutil
from pathlib import Path

repo = Path().resolve().parent           # 请在 tutorials/ 目录下运行本笔记本
out = Path.cwd() / "quickstart_output_pathd8"     # 位于 tutorials/ 内
shutil.rmtree(out, ignore_errors=True)

cmd = [
    sys.executable, "-m", "phylodater", "dating",
    "-t", str(repo / "examples" / "tree.nwk"),
    "-s", str(repo / "examples" / "alignment.fasta"),
    "-c", str(repo / "examples" / "calibrations.yaml"),
    "-o", str(out),
    "--method", "pathd8",
    "--seed", "42",
]
print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=repo)
print("exit code:", result.returncode)
print(result.stdout[-1500:])


### 这次运行写出了什么

每个方法都产出同一套标准化结果：一棵定年后的树、一个 `comparison_table.tsv`
（每行一个节点，每个方法一组列）、一个 `summary.txt`、一个 `comparison_plot.png`，
以及一个 `runtime_metadata.json`，其中记录耗时、告警，以及产出这些数值时所用的**引擎版本**。

In [ ]:

import csv
from pathlib import Path

out = Path.cwd() / "quickstart_output_pathd8"
print("files written by the run:")
for p in sorted(out.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(out))

table = out / "comparison_table.tsv"
if table.exists():
    rows = list(csv.DictReader(table.open(), delimiter="\t"))
    print("\nnode ages reported by PATHd8:")
    for r in rows[:8]:
        mean = r.get("pathd8_mean", "")
        print(f"  {r['Node']:<18} {mean}")


## 3. Python API

同一条流水线也能在进程内调用，这正是你在更大的工作流中想要的用法。
`PipelineConfig(methods=[...])` 接受多个引擎，并把它们并行运行。

In [ ]:

import sys
from pathlib import Path

repo = Path().resolve().parent            # 请在 tutorials/ 目录下运行本笔记本
sys.path.insert(0, str(repo))             # PhyloDater 经 pip 安装后这行是空操作

from phylodater import PhylogeneticTree, CalibrationLoader
from phylodater.core import DatingPipeline, PipelineConfig

import shutil
_LIB_OUT = Path.cwd() / "quickstart_output_library"      # 位于 tutorials/ 内
shutil.rmtree(_LIB_OUT, ignore_errors=True)

tree = PhylogeneticTree.from_file(repo / "examples" / "tree.nwk")
calibrations = CalibrationLoader().load(repo / "examples" / "calibrations.yaml")
print(f"{len(tree.tip_names)} tips, {len(calibrations)} calibrations")

config = PipelineConfig(
    methods=["pathd8"],
    output_dir=_LIB_OUT,
    seed=42,
    save_intermediates=False,
    enable_checkpoint=False,
)
pipeline = DatingPipeline(config)
results = pipeline.run(tree, repo / "examples" / "alignment.fasta", calibrations)

for name, res in results.items():
    print(f"{name}: {len(res.node_ages)} node ages, "
          f"root = {max((a.mean_age for a in res.node_ages.values()), default=float('nan')):.2f} Ma")


## 4. 对照真值核查精度

“它跑通了吗？”并不是那个有意思的问题——有意思的是“这棵时间树对不对？”。
`test-data/benchmark/` 是一个严格钟的 JC69 模拟数据集，其真实节点年龄记录在 `truth.yaml` 里，因此每个方法都能被数值化打分。
`scripts/run_reference_benchmark.py` 就是做这件事的，它依次检查完成度、末端节点完整性、拓扑保持、时序、校准是否被遵守，以及精度。

把真实的化石校准读作真值会陷入循环论证——喂给方法的正是这些数字——这正是参考数据集采用模拟数据的原因。
设计与验收门禁见 [docs/TESTING_CN.md](../docs/TESTING_CN.md)；实测数值见 [docs/TEST_RESULTS_CN.md](../docs/TEST_RESULTS_CN.md)。

In [ ]:

import shutil, subprocess, sys
from pathlib import Path

repo = Path().resolve().parent
out = Path.cwd() / "quickstart_benchmark"               # 位于 tutorials/ 内
shutil.rmtree(out, ignore_errors=True)
shutil.rmtree(out, ignore_errors=True)

# 对照随包发布的仿真真值给两个方法打分
result = subprocess.run(
    [
        sys.executable, str(repo / "scripts" / "run_reference_benchmark.py"),
        "--dataset", str(repo / "test-data" / "benchmark"),
        "--output", str(out),
        "--methods", "pathd8", "lsd2",
    ],
    capture_output=True, text=True, cwd=repo,
)
print(result.stdout[-2500:] or result.stderr[-2500:])


## 5. 哪些方法可以在本环境运行

测试和 CLI 遇到没有安装的引擎会直接跳过，而不是假装通过，所以先检查一遍环境是值得的。

In [ ]:

import importlib.util, shutil

from phylodater.core import DatingMethodRegistry

backend = {
    "mcmctree": ["mcmctree", "baseml"],
    "lsd2": ["iqtree2", "iqtree"],
    "pathd8": ["PATHd8", "pathd8"],
    "treepl": ["treepl", "treePL"],
    "r8s": ["r8s"],
    "wlogdate": ["launch_wLogDate.py"],
    "mdcat": ["mdcat", "md_cat.py"],
}
python_backend = {
    "r8s": "pyr8s",
    "wlogdate": "logdate",
    "mdcat": "emd",
}

print(f"{'method':<10} {'in registry':<12} backend found")
for method in DatingMethodRegistry.list_methods():
    binary = any(shutil.which(b) for b in backend.get(method, []))
    package = importlib.util.find_spec(python_backend.get(method, "")) is not None \
        if method in python_backend else False
    print(f"{method:<10} {'yes':<12} {'yes' if (binary or package) else 'no  (tests skip)'}")


## 6. 延伸阅读

- [docs/TESTING_CN.md](../docs/TESTING_CN.md) —— 测试方法与验收门禁
- [docs/TEST_RESULTS_CN.md](../docs/TEST_RESULTS_CN.md) —— 七个引擎的全部实测结果
- [docs/EXTERNAL_TOOLS_CN.md](../docs/EXTERNAL_TOOLS_CN.md) —— 引擎安装与溯源
- [docs/ERROR_CODES_CN.md](../docs/ERROR_CODES_CN.md) —— 退出码与错误类别
- [docs/ARCHITECTURE_CN.md](../docs/ARCHITECTURE_CN.md) —— 本包的组装方式
- `phylodater dating --help` —— 全部选项，包括用于引擎专属参数的 `--method-args`

### 故障排查

| 症状 | 原因与修复 |
|------|------------|
| 某个方法报 `Environment not available` | `PATH` 里没有这个引擎；请装上它（[docs/EXTERNAL_TOOLS_CN.md](../docs/EXTERNAL_TOOLS_CN.md)），或改用别的 `--method` |
| LSD2 报 `ERROR: File not found LG` | 为核苷酸比对请求了氨基酸模型；请取消 `model` 设置（auto），或传入 `--method-args model=GTR+G` |
| 告警里点到你的某个校准 | 该引擎无法表达这条约束（语义降级）——请读一下告警，它写明了引擎实际拿到的是什么 |
| MD-Cat 跑了几个小时 | 你启用了 `ci_nboots`；它默认为 `0`，因为单次 bootstrap 重复的代价很高 |